# Day 6 — Logistic regression from scratch: sigmoid, cross-entropy, and a real gradient bug

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

## Setup — reuse Day 2's Titanic cleaning

Same feature set as Day 2 (`pclass`, `fare`, `who`, `family_size`), but kept **unscaled** for now — that's the point. `fare` runs 0 to 512; the others are single digits. We'll feel the consequence of that mismatch in Step 4.

In [ ]:
df = sns.load_dataset("titanic")
df["sex"] = df["sex"].map({"male": 0, "female": 1})
df["embarked"] = df["embarked"].map({"C": 0, "Q": 1, "S": 2})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df.drop(
    columns=[
        "class",
        "embark_town",
        "alive",
        "alone",
        "sibsp",
        "parch",
        "deck",
        "adult_male",
    ],
    inplace=True,
)

median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age)
mode_embarked = df["embarked"].mode()[0]
df["embarked"] = df["embarked"].fillna(mode_embarked)
df["embarked"] = df["embarked"].astype(int)
df["who"] = df["who"].map({"man": 0, "woman": 1, "child": 2})
df.drop(["sex", "age", "embarked"], axis=1, inplace=True)

x_train, x_test, y_train, y_test = train_test_split(
    df.drop("survived", axis=1), df["survived"], test_size=0.2, random_state=42
)

# raw numpy views — this is what the from-scratch functions below operate on
X_train = x_train.values.astype(float)
X_test = x_test.values.astype(float)
y_train_v = y_train.values.astype(float)
y_test_v = y_test.values.astype(float)

x_train.describe()

## Step 1 — sigmoid sanity checks

`sigmoid(z) = 1 / (1 + e^-z)` maps any raw score to a probability. Check the three landmark points: `z=0` should sit exactly at the midpoint, and large positive/negative `z` should saturate toward 1/0.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))


print("sigmoid(0):  ", sigmoid(0))
print("sigmoid(10): ", sigmoid(10))
print("sigmoid(-10):", sigmoid(-10))

## Step 2 — cross-entropy, verified against a real fitted model

Before trusting `cross_entropy` inside a gradient loop, check it against `sklearn.metrics.log_loss` on real predicted probabilities. Fit a quick `LogisticRegression` (Day 2 style — `RobustScaler` on `fare` only, since fare is right-skewed with genuine outliers) and compare.

In [ ]:
def cross_entropy(y_true, y_pred, eps=1e-15):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))


scaler = RobustScaler()
x_train_scaled = x_train.copy()
x_test_scaled = x_test.copy()
x_train_scaled[["fare"]] = scaler.fit_transform(x_train[["fare"]])
x_test_scaled[["fare"]] = scaler.transform(x_test[["fare"]])

model = LogisticRegression()
model.fit(x_train_scaled, y_train)

probs_test = model.predict_proba(x_test_scaled)[:, 1]
manual_ce = cross_entropy(y_test_v, probs_test)
sk_ll = log_loss(y_test_v, probs_test)

print("manual cross-entropy:", manual_ce)
print("sklearn log_loss:    ", sk_ll)
print("match:", np.isclose(manual_ce, sk_ll))

## Step 3 — implement the gradient

For logistic regression + cross-entropy, the chain rule collapses to a clean form: `dw = (1/n) X^T (p - y)`, `db = mean(p - y)`. `error = p - y` is how wrong each prediction is; `dw` is just that error, averaged per feature.

In [ ]:
def gradients_logreg(w, b, X, y):
    n = len(y)
    p = sigmoid(X @ w + b)
    error = p - y
    dw = (1 / n) * (X.T @ error)
    db = (1 / n) * np.sum(error)
    return dw, db


def cost_logreg(w, b, X, y, eps=1e-15):
    p = sigmoid(X @ w + b)
    return cross_entropy(y, p, eps=eps)

## Step 4 — gradient check, and it fails

Same numerical-vs-analytical check as Day 1/Day 5's from-scratch models, run on the **unscaled** features (`pclass`, `fare`, `who`, `family_size` straight from the train split, no scaling).

In [ ]:
rng = np.random.default_rng(42)
w_test = rng.normal(0, 0.1, size=X_train.shape[1])  # arbitrary small random weights
b_test = 0.0
eps = 1e-4  # tiny nudge for the finite-difference approximation

dw_analytical, db_analytical = gradients_logreg(w_test, b_test, X_train, y_train_v)

dw_numerical = np.zeros_like(w_test)
for i in range(len(w_test)):
    w_plus = w_test.copy()
    w_plus[i] += eps
    w_minus = w_test.copy()
    w_minus[i] -= eps
    dw_numerical[i] = (
        cost_logreg(w_plus, b_test, X_train, y_train_v)
        - cost_logreg(w_minus, b_test, X_train, y_train_v)
    ) / (2 * eps)

print("dw analytical:", dw_analytical)
print("dw numerical: ", dw_numerical)
print("max abs diff: ", np.max(np.abs(dw_analytical - dw_numerical)))

## Step 5 — diagnose it

The `fare` gradient is the outlier — off by ~2, stable across `eps` from `1e-4` down to `1e-7` (checked separately), so it's not finite-difference noise. Check the raw scores `z = Xw + b` the model is actually producing.

In [ ]:
z = X_train @ w_test + b_test
print("z min:", z.min(), " z max:", z.max())
print("sigmoid(z.min()):", sigmoid(z.min()))

# fare reaches 512 unscaled — even a small random weight on it dominates z and
# pushes some samples to z ~ -53. sigmoid(-53) sits at the edge of float64
# precision: the analytical gradient works with that raw probability directly,
# while the numerical check computes cost(w+eps) - cost(w-eps) — a tiny
# difference between two numbers both already squeezed near a precision floor.
# Subtracting them loses precision (catastrophic cancellation) — a real
# numerical bug, not a wrong formula.

## Step 6 — fix it, verified

Scale the numeric columns — fit on train only, same leakage discipline as every day this week. Then rerun the exact same gradient check.

In [ ]:
std_scaler = StandardScaler()
X_train_std = std_scaler.fit_transform(X_train)

z_std = X_train_std @ w_test + b_test
print("z range (scaled):", z_std.min(), "to", z_std.max())

dw_analytical2, db_analytical2 = gradients_logreg(
    w_test, b_test, X_train_std, y_train_v
)

dw_numerical2 = np.zeros_like(w_test)
for i in range(len(w_test)):
    w_plus = w_test.copy()
    w_plus[i] += eps
    w_minus = w_test.copy()
    w_minus[i] -= eps
    dw_numerical2[i] = (
        cost_logreg(w_plus, b_test, X_train_std, y_train_v)
        - cost_logreg(w_minus, b_test, X_train_std, y_train_v)
    ) / (2 * eps)

print("dw analytical:", dw_analytical2)
print("dw numerical: ", dw_numerical2)
print("max abs diff: ", np.max(np.abs(dw_analytical2 - dw_numerical2)))